### Empezamos con la Dimensión de conductores.

Hay algunas tablas que su generación de datos estuvieron correctamente estandarizados, por lo que se podría saltar a hacer la dimensión casi directamente.

In [29]:
#dimension de conductores
from pyspark.sql import functions as F
from pyspark.sql.window import Window


conductores = spark.read.table("silver.silver_ope_conductores")

conductores = conductores.withColumn(
    "Antiguedad_anios",
    F.floor(
        F.months_between(F.current_date(), F.col("fec_ingreso"))/12
    )
)

#los datos fueron generados están estandarizados, asi que:
#hacemos la dimensión
dim_conductores = conductores.select(
    "cond_id",
    "nomb_cond", 
    "apell_cond", 
    "tip_doc", 
    "num_doc_hash",
    "fec_ingreso", 
    "antiguedad_anios", 
    "id_ciudad_base", 
    "tip_vehiculo",
    "cod_zona_asignada", 
    "activo", 
    "calific_promedio_acum"
)

w = Window.orderBy("cond_id")

#creamos una surrogate key y la organizamos de primera
dim_conductores = dim_conductores.withColumn("sk_conductor",F.row_number().over(w))
dim_conductores = dim_conductores.select("sk_conductor", *[c for c in dim_conductores.columns if c != "sk_conductor"])

dim_conductores = dim_conductores.select(
    "sk_conductor",
    "cond_id",
    "nomb_cond", 
    "apell_cond", 
    "tip_doc", 
    "num_doc_hash",
    "fec_ingreso", 
    "antiguedad_anios", 
    "id_ciudad_base", 
    "tip_vehiculo",
    "cod_zona_asignada", 
    "activo", 
    "calific_promedio_acum"
)

display(dim_conductores.limit(5))
print("guardando en gold...")
dim_conductores.write.format("delta").mode("overwrite").saveAsTable("gold.dim_conductores")
print("guardado")

StatementMeta(, 1e8d28f2-61c4-4e24-ae05-d1364b6980a5, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7c95a86c-de1f-44ba-bef6-e2335e387308)

guardando en gold...
guardado


### Proseguimos con la dimensión de remitentes.

In [30]:
from pyspark.sql.types import *

remitentes = spark.read.table("silver.silver_cli_remitentes")
#estandarizamos
remitentes = remitentes.withColumn(
    "segmento_industria",
    F.when(F.lower(F.col("tipo_cliente")).contains("e-commerce"), "Ecommerce")
    .when(F.lower(F.col("tipo_cliente")).contains("corporativo"), "Corporativo")
    .when(F.lower(F.col("tipo_cliente")).contains("natural"), "Natural")
    .otherwise("Otro"))

print(remitentes.count())
#estandarizamos SLA
remitentes = remitentes.withColumn("sla_entrega_horas", F.col("sla_entrega_horas").cast(IntegerType()))

#creamos la dimensión
dim_remitentes = remitentes.select(
    "id_remitente",
    "razon_social",
    "segmento_industria",
    "ciudad_principal",
    "sla_entrega_horas",
    "penalidad_porc",
    "activo"
)

#sugorrate key
w = Window.orderBy("id_remitente")
dim_remitentes = dim_remitentes.withColumn(
    "sk_remitente",
    F.row_number().over(w)
)

dim_remitentes = dim_remitentes.select(
    "sk_remitente",
    "id_remitente",
    "razon_social",
    "segmento_industria",
    "ciudad_principal",
    "sla_entrega_horas",
    "penalidad_porc",
    "activo"
)
display(dim_remitentes.limit(5))
print("guardando en gold...")
dim_remitentes.write.format("delta").mode("overwrite").saveAsTable("gold.dim_remitentes")
print("guardado")

StatementMeta(, 1e8d28f2-61c4-4e24-ae05-d1364b6980a5, 32, Finished, Available, Finished, False)

200


SynapseWidget(Synapse.DataFrame, e15449a4-b667-463e-bdcd-64af86570a4c)

guardando en gold...
guardado


### Luego la Dimensión de Zonas
se tuvieron que crear varias columnas nuevas para poder hacer los cálculos y clasificaciones.

In [50]:
from pyspark.sql import functions as F

# 1. Leer tabla origen
zonas = spark.read.table("silver.silver_geo_zonas")

# 2. Enriquecer con nombres de municipios y calcular variables en un solo flujo
zonas_procesado = (
    zonas
    # Homologación de municipios
    .withColumn(
        "municipio",
        F.when(F.col("id_ciudad") == "BOG", "Bogotá")
        .when(F.col("id_ciudad") == "CAL", "Cali")
        .when(F.col("id_ciudad") == "MDE", "Medellin")
        .when(F.col("id_ciudad") == "BAQ", "Barranquilla")
        .when(F.col("id_ciudad") == "BGA", "Bucaramanga")
        .when(F.col("id_ciudad") == "PER", "Pereira")
        .when(F.col("id_ciudad") == "MZL", "Manizales")
        .when(F.col("id_ciudad") == "CTG", "Cartagena")
        .when(F.col("id_ciudad") == "SMR", "Santa Marta")
        .when(F.col("id_ciudad") == "CUC", "Cúcuta")
    )
    # V1: Tráfico promedio (Corregidos los rangos continuos sin huecos)
    .withColumn(
        "V1_trafico",
        F.when(F.col("nivel_trafico_prom") < 20, 1)
        .when((F.col("nivel_trafico_prom") >= 20) & (F.col("nivel_trafico_prom") < 30), 2)
        .when((F.col("nivel_trafico_prom") >= 30) & (F.col("nivel_trafico_prom") < 40), 3) # Se cambió 35 por 40 para evitar saltos
        .when((F.col("nivel_trafico_prom") >= 40) & (F.col("nivel_trafico_prom") < 45), 4)
        .when(F.col("nivel_trafico_prom") >= 45, 5) # Simplificado para atrapar todo lo mayor a 45
        .otherwise(3)
    )
    # V2: Distancia (Corregido 'distancia_km' por 'distancia_bodega_km')
    .withColumn(
        "V2_distancia",
        F.when(F.col("distancia_bodega_km") < 5.0, 1.0)
        .when((F.col("distancia_bodega_km") >= 5.0) & (F.col("distancia_bodega_km") < 15.0), 2.0)
        .when((F.col("distancia_bodega_km") >= 15.0) & (F.col("distancia_bodega_km") < 40.0), 3.0)
        .when((F.col("distancia_bodega_km") >= 40.0) & (F.col("distancia_bodega_km") <= 80.0), 4.0)
        .when(F.col("distancia_bodega_km") > 80.0, 5.0)
        .otherwise(3.0)
    )
    # Cálculo del índice de dificultad operativa
    .withColumn(
        "indice_dificultad_operativa", 
        (F.col("V1_trafico") * 0.60) + (F.col("V2_distancia") * 0.40)
    )
)

dim_zonas = zonas_procesado.select(
    "id_zona",
    "nom_zona",
    "municipio",
    "barrio_referencia",
    "latitud_centroide",
    "longitud_centroide",
    "nivel_trafico_prom",
    "distancia_bodega_km",
    "indice_dificultad_operativa",
    "tip_zona"
)

#sugorrate key

w = Window.orderBy("id_zona")

dim_zonas = dim_zonas.withColumn(
    "sk_zona",
    F.row_number().over(w)
)

dim_zonas = dim_zonas.select(
    "sk_zona",
    "id_zona",
    "nom_zona",
    "municipio",
    "barrio_referencia",
    "latitud_centroide",
    "longitud_centroide",
    "nivel_trafico_prom",
    "distancia_bodega_km",
    "indice_dificultad_operativa",
    "tip_zona"
)

# 3. Visualizar el resultado final
display(dim_zonas.limit(5))
print("guardando en gold...")
dim_zonas.write.format("delta").mode("overwrite").saveAsTable("gold.dim_zonas")
print("guardado")

StatementMeta(, 1e8d28f2-61c4-4e24-ae05-d1364b6980a5, 52, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 56820633-86ca-4bba-8aca-a22be156ff32)

guardando en gold...
guardado


### Sigue la primera tabla de hechos, la fact_envios
Durante el desarollo se pueden crear las sugorrate keys para el sistema, pero es opcional.

In [80]:
envios = spark.read.table("silver.silver_tms_envios")
remitentes = spark.read.table("gold.dim_remitentes")
#join para traer sla_entrega_horas 
fact_envios = (envios.alias("e").join(remitentes.select("id_remitente", "sla_entrega_horas").alias("r"), on="id_remitente", how="left"))

#tiempo de entrega
fact_envios = fact_envios.withColumn("tiempo_entrega_real_horas",(F.unix_timestamp("fec_entrega_real") - F.unix_timestamp("fec_recepcion")) / 3600)

#cumplimiento de sla
fact_envios = fact_envios.withColumn(
    "flag_cumplimiento_sla",
    F.when(F.col("estado_final") != "Entregado", "NO ENTREGADO")
    .when(F.col("tiempo_entrega_real_horas") <= F.col("sla_entrega_horas"), "SI")
    .otherwise("NO"))

#suma de intentos
fact_envios = fact_envios.withColumn("numero_intentos",
    F.when(F.col("fec_intento1").isNotNull(), 1).otherwise(0) + F.when(F.col("fec_intento2").isNotNull(), 1).otherwise(0)
)

#clasificación horas de retraso
fact_envios = fact_envios.withColumn("horas_retraso", (F.unix_timestamp("fec_entrega_real") - F.unix_timestamp("fec_entrega_programada")) / 3600)
fact_envios = fact_envios.withColumn(
    "clasificacion_retraso",
    F.when(F.col("estado_final") != "Entregado", "No entregado")
    .when(F.col("horas_retraso") <=0 , "A tiempo")
    .when(F.col("horas_retraso") <=4 , "Retraso leve")
    .when(F.col("horas_retraso") <=24 , "Retraso moderado")
    .otherwise("Retraso crítico")
)


fact_envios = fact_envios.select(
    "id_envio",
    "id_remitente",
    "cond_id",
    "id_zona_destino",
    "fec_recepcion",
    "fec_entrega_programada",
    "fec_entrega_real",
    "estado_final",
    "sla_entrega_horas",
    "tiempo_entrega_real_horas",
    "horas_retraso",
    "flag_cumplimiento_sla",
    "clasificacion_retraso",
    "numero_intentos",
    "motivo_fallo_cod",
    "tip_paquete",
    "peso_kg",
    "vr_declarado"

)


w = Window.orderBy("id_envio")

fact_envios = fact_envios.withColumn(
    "sk_envio",
    F.row_number().over(w)
)

fact_envios = fact_envios.select(
    "sk_envio",
    "id_envio",
    "id_remitente",
    "cond_id",
    "id_zona_destino",
    "fec_recepcion",
    "fec_entrega_programada",
    "fec_entrega_real",
    "estado_final",
    "sla_entrega_horas",
    "tiempo_entrega_real_horas",
    "horas_retraso",
    "flag_cumplimiento_sla",
    "clasificacion_retraso",
    "numero_intentos",
    "motivo_fallo_cod",
    "tip_paquete",
    "peso_kg",
    "vr_declarado"
)

display(fact_envios.limit(10))
print("guardando en gold...")
fact_envios.write.format("delta").mode("overwrite").saveAsTable("gold.fact_envios")
print("guardado")

StatementMeta(, 1e8d28f2-61c4-4e24-ae05-d1364b6980a5, 82, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6e2994c0-43e4-4334-8466-5e0742272221)

guardando en gold...
guardado


### Continuamos con la fact de rutas

In [83]:
rutas = spark.read.table("silver.silver_gps_rutas")

rutas = rutas.withColumn("eficiencia_ruta",
    F.when(F.col("num_paradas_plan") > 0, F.round(F.col("num_paradas_real") / F.col("num_paradas_plan"), 2)).otherwise(None)
)

rutas = rutas.withColumn("horas_trabajadas", 
    F.when(F.col("hra_fin") > F.col("hra_inicio"), F.round((F.unix_timestamp("hra_fin") - F.unix_timestamp("hra_inicio"))/ 3600, 2 )).otherwise(0))

rutas = rutas.withColumn("velocidad_promedio_kmh",
    F.when(F.col("horas_trabajadas") > 0, F.round(F.col("km_recorridos")/ F.col("horas_trabajadas"),2)).otherwise(None)
)

rutas = rutas.withColumn(
    "desviacion_porc", F.when( F.col("km_recorridos") > 0, F.round( (F.col("desviacion_ruta_km") / F.col("km_recorridos")) * 100, 2)).otherwise(None)
)

fact_rutas = rutas.select(
    "id_ruta",
    "cond_id",
    "fec_ruta",
    "hra_inicio",
    "hra_fin",
    "km_recorridos",
    "horas_trabajadas",
    "num_paradas_plan",
    "num_paradas_real",
    "desviacion_ruta_km",
    "eficiencia_ruta",
    "velocidad_promedio_kmh",
    "desviacion_porc"
)

w = Window.orderBy("id_ruta")

fact_rutas = fact_rutas.withColumn(
    "sk_ruta",
    F.row_number().over(w)
)

fact_rutas = fact_rutas.select(
    "sk_ruta",
    "id_ruta",
    "cond_id",
    "fec_ruta",
    "hra_inicio",
    "hra_fin",
    "km_recorridos",
    "horas_trabajadas",
    "num_paradas_plan",
    "num_paradas_real",
    "desviacion_ruta_km",
    "eficiencia_ruta",
    "velocidad_promedio_kmh",
    "desviacion_porc"
)

display(fact_rutas.limit(5))
print("guardando en gold...")
fact_rutas.write.format("delta").mode("overwrite").saveAsTable("gold.fact_rutas")
print("guardado")

StatementMeta(, 1e8d28f2-61c4-4e24-ae05-d1364b6980a5, 85, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c44c9195-8de3-453b-b6c5-988550e9edee)

guardando en gold...
guardado


### Creamos la fact de desempeño del conductor
Esta tiene una mayor dificultad lógica que las anteriores, pero se intentó que un mismo bloque se hicieran los joins, los cálculos y la selección de los resultados.

In [2]:
from pyspark.sql import functions as F

envios = spark.read.table("gold.fact_envios")
rutas = spark.read.table("gold.fact_rutas")
dest = spark.read.table("silver.silver_cal_destinatarios")

fact_desempeno = (
    envios
    .join(rutas.select("cond_id","eficiencia_ruta","velocidad_promedio_kmh"), "cond_id", "left")
    .join(dest.select("id_envio","puntaje_1_5"), "id_envio", "left")
    .groupBy("cond_id")
    .agg(
        F.count("*").alias("total_envios"),
        F.sum(F.when(F.col("estado_final")=="Entregado",1).otherwise(0)).alias("envios_exitosos"),
        F.avg("numero_intentos").alias("promedio_intentos"),
        F.avg("eficiencia_ruta").alias("adherencia_ruta"),
        F.avg("velocidad_promedio_kmh").alias("velocidad_promedio"),
        F.avg("puntaje_1_5").alias("calificacion_promedio")
    )
    .withColumn("tasa_exito", F.round(F.col("envios_exitosos")/F.col("total_envios"),4))
    .withColumn("score_desempeno",
        F.round(
            F.col("tasa_exito")*0.35 +
            F.least(F.col("adherencia_ruta"),F.lit(1))*0.20 + (F.col("velocidad_promedio")/120)*0.20 + 
            (1/(F.col("promedio_intentos")+1))*0.15 + (F.col("calificacion_promedio")/5)*0.10,2)
        )
    .select(
        "cond_id", "total_envios", "envios_exitosos", "tasa_exito", "promedio_intentos", "adherencia_ruta", "velocidad_promedio", "calificacion_promedio", "score_desempeno"
    )
)

fact_desempeno.write.format("delta").mode("overwrite").saveAsTable("gold.fact_desempeno_conductor")

display(fact_desempeno.limit(10))

StatementMeta(, 743724b6-4eaa-46fe-85d0-4d13ff45cd8f, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e73f6207-2e70-4f6f-b08e-90aa1d7accdc)

### fact de trazabilidad del envio
Se sabe la importancia de los ajustes de tiempo y los historicos de los datos en esta área de trabajo , asi que tener esta tabla de hechos es algo muy correcto

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

envios = spark.read.table("gold.fact_envios")
novedades = spark.read.table("silver.silver_dir_novedades")

# Eventos provenientes de TMS
recepcion = envios.select(
    "id_envio",
    F.lit("Recepción").alias("evento"),
    F.lit(None).cast("string").alias("descripcion"),
    F.col("fec_recepcion").cast("timestamp").alias("fecha_evento")
)

programada = envios.select(
    "id_envio",
    F.lit("Entrega programada").alias("evento"),
    F.lit(None).cast("string").alias("descripcion"),
    F.col("fec_entrega_programada").cast("timestamp").alias("fecha_evento")
)

entrega = envios.select(
    "id_envio",
    F.lit("Entrega real").alias("evento"),
    F.lit(None).cast("string").alias("descripcion"),
    F.col("fec_entrega_real").cast("timestamp").alias("fecha_evento")
)

# Eventos provenientes de novedades
eventos_novedades = novedades.select(
    "id_envio",
    F.col("tip_novedad").alias("evento"),
    F.col("desc_novedad").alias("descripcion"),
    F.col("fec_novedad").cast("timestamp").alias("fecha_evento")
)

# Línea de tiempo
fact_trazabilidad = (
    recepcion
    .unionByName(programada)
    .unionByName(entrega)
    .unionByName(eventos_novedades)
    .filter(F.col("fecha_evento").isNotNull())
)

# Orden cronológico y tiempo entre eventos
w = Window.partitionBy("id_envio").orderBy("fecha_evento")

fact_trazabilidad = (
    fact_trazabilidad
    .withColumn("evento_anterior", F.lag("evento").over(w))
    .withColumn("fecha_anterior", F.lag("fecha_evento").over(w))
    .withColumn("horas_desde_evento_anterior", F.round((F.unix_timestamp("fecha_evento") - F.unix_timestamp("fecha_anterior"))/3600, 2))
    .drop("fecha_anterior")
)

fact_trazabilidad.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.fact_trazabilidad_envio")

display(fact_trazabilidad)

StatementMeta(, 743724b6-4eaa-46fe-85d0-4d13ff45cd8f, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a4f147e6-6c2c-4ca2-90a9-8aac3b018e15)